# 03 DistilBERT (Transformer)

**Goal:** See whether a fine-tuned transformer can beat the Logistic Regression baselines from notebook 02. DistilBERT understands word context, not just word presence, so this is the test of whether context actually helps on this dataset — and whether the accuracy gain (if any) is worth the much higher compute cost.

This notebook runs on Google Colab with a GPU, since fine-tuning a transformer on CPU is impractically slow.


## Check GPU

DistilBERT needs GPU to train in reasonable time.

This should print `True` — if not, enable it via Runtime → Change runtime type → T4 GPU.

In [1]:
import torch
print('GPU available:', torch.cuda.is_available())

GPU available: True


## Upload the dataset

Colab can't see my local files, so I upload the CSV directly. (This is cleared when the session ends, so it has to be re-uploaded each new session.)

In [2]:
from google.colab import files
uploaded = files.upload()

Saving enron_spam_data.csv to enron_spam_data.csv


In [3]:
import pandas as pd
df = pd.read_csv('enron_spam_data.csv')
df.shape

(33716, 5)

## Preprocess and remove duplicates

Same preprocessing as notebook 02: fill empty fields, merge subject + body, encode the label.

I also remove duplicate emails **before** splitting — and this step came directly from a sanity check. My first training runs scored ~0.99, which was high enough to make me suspicious of data leakage.

Checking for duplicates revealed that **9.6% of the emails were exact repeats** — a real risk, since duplicates split across train and test would let the model "predict" emails it had already seen.

I removed them and re-ran. The scores held, which told me the high performance wasn't leakage but a reflection of how separable this dataset genuinely is.

Removing duplicates up front keeps every split below honest.

In [4]:
# Fill empty fields, merge subject + body, encode the label
df['Subject'] = df['Subject'].fillna('')
df['Message'] = df['Message'].fillna('')
df['text'] = df['Subject'] + ' ' + df['Message']
df['label'] = (df['Spam/Ham'] == 'spam').astype(int)

# Drop duplicate emails before splitting
df_unique = df.drop_duplicates(subset='text').reset_index(drop=True)
print("Before:", len(df), "-> after removing duplicates:", len(df_unique))

Before: 33716 -> after removing duplicates: 30494


## Train / validation / test split

Same 70/15/15 split and seed as notebook 02, so the transformer is evaluated on a comparable test set.

In [5]:
from sklearn.model_selection import train_test_split

train_val, test = train_test_split(
    df,
    test_size=0.15,
    random_state=42,
    stratify=df['label']
)

train, val = train_test_split(
    train_val,
    test_size=0.176,
    random_state=42,
    stratify=train_val['label']

)

print(f"Split done — train {len(train)}, val {len(val)}, test {len(test)}")

Split done — train 23614, val 5044, test 5058


## Load DistilBERT

`distilbert-base-uncased` is a smaller,faster version of BERT,already pre-trained on large text corpus.

I load it with fresh 2-class classification head (spam / ham) and fine-tune just that for the task.

The "newly initialized" warning is expected — that's the classification head I'm about to train.

In [6]:
# Hugging Face libraries for loading and fine-tuning DistilBERT
!pip install -q transformers datasets

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "distilbert-base-uncased"

# Tokenizer turns text into numbers the model understands
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load DistilBERT with a fresh classification head (2 classes: spam / ham)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

print("Model loaded:", model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: distilbert-base-uncased


## Tokenize the text

The tokenizer transforms raw email text into BERT-compatible token IDs.

Inputs are truncated to 256 tokens to balance information retention and computational efficiency, while shorter sequences are padded to a fixed length for uniform batch processing.

The tokenized outputs,along with attention masks, are returned as PyTorch tensors for downstream training and evaluation.


In [8]:
def tokenize(texts):
    return tokenizer(
        list(texts),
        truncation = True,
        max_length = 256,
        padding = True,
        return_tensors = "pt"
    )
train_enc = tokenize(train['text'])
val_enc = tokenize(val['text'])

print("Tokenized — train:", train_enc['input_ids'].shape)


Tokenized — train: torch.Size([23614, 256])


## Wrap data in a Dataset

The Hugging Face `Trainer` expects a `Dataset` that can return one (inputs + label) pair at time.

This small wrapper does exactly that:  `__len__` reports the count,
`__getitem__` returns the tokens and label for a given index.

In [9]:
import torch

class SpamDataset(torch.utils.data.Dataset):
  def __init__(self,encodings,labels):
    self.encoding = encodings
    self.labels = list(labels)

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, indx):
     item = {k: v[indx] for k,v in self.encoding.items()}
     item['labels'] =torch.tensor(self.labels[indx])
     return item

train_dataset = SpamDataset(train_enc,train['label'])
val_dataset = SpamDataset(val_enc,val['label'])

print("Dataset ready:", len(train_dataset),"train,",len(val_dataset),"val")

Dataset ready: 23614 train, 5044 val


## Training setup

I fine-tune for just one epoch — since DistilBERT is already pretrained on a large English corpus, it only needs light task-specific adaptation, and more epochs would risk overfitting for little gain.

I use a batch size of 16, which balances speed and memory and fits comfortably on a free NVIDIA T4 GPU while keeping training stable.

In [10]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs =1,
    per_device_train_batch_size =16,
    per_device_eval_batch_size =32,
    eval_strategy ="epoch",
    logging_steps=50,
    report_to ="none"
)

## Metrics function

The `Trainer` calls this after evaluation.
It turns the model's raw scores(logits) into predicted labels with `argmax`, then computes the same four metrics I used in notebook 02.

In [11]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'precision': precision_score(labels, predictions),
        'recall': recall_score(labels, predictions),
        'f1': f1_score(labels, predictions)
    }

## Fine-tunning

Assemble the `Trainer` with the model, settings, data, and metrics, then train.
This takes ~11 minutes on a T4 GPU.
Watch the training loss drop over steps — that's the model learning to separate spam from ham.

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.031771,0.025202,0.993656,0.993388,0.994161,0.993774


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1476, training_loss=0.05832021164538737, metrics={'train_runtime': 703.7476, 'train_samples_per_second': 33.555, 'train_steps_per_second': 2.097, 'total_flos': 1564042575931392.0, 'train_loss': 0.05832021164538737, 'epoch': 1.0})

## Test-set evaluation + inference time

Final evaluation on the held-out test set, plus how long prediction takes per email.

The per-email time is the key contrast with notebook 02: DistilBERT is far more expensive to run, so its accuracy edge has to justify that cost.

In [13]:
import time

# Tokenize and wrap the test set the same way as train/val
test_encodings = tokenize(test['text'])
test_dataset = SpamDataset(test_encodings, test['label'])

# Time prediction over the whole test set
start = time.perf_counter()
pred_output = trainer.predict(test_dataset)
inference_time = time.perf_counter() - start

test_preds = np.argmax(pred_output.predictions, axis=-1)
n = len(test)

print("=== DistilBERT TEST RESULTS ===")
print(f"Accuracy: {accuracy_score(test['label'], test_preds):.4f} | "
      f"Precision: {precision_score(test['label'], test_preds):.4f} | "
      f"Recall: {recall_score(test['label'], test_preds):.4f} | "
      f"F1 {f1_score(test['label'], test_preds):.4f} | "
      f"{inference_time/n*1000:.1f} ms/email")

=== DistilBERT TEST RESULTS ===
Accuracy: 0.9957 | Precision: 0.9953 | Recall: 0.9961 | F1 0.9957 | 8.7 ms/email


## Multi-seed validation

To check that the results aren't driven by a single lucky split, I repeat fine-tuning across three random seeds. Each run re-splits the data and starts from a fresh DistilBERT (pretrained weights, newly initialized classification head), trained independently.

I report the mean ± standard deviation so the spread is visible, not just a single number.

In [14]:
seeds = [0, 1, 2]
bert_scores = []  # one (accuracy, precision, recall, f1) row per seed

for seed in seeds:
    print(f"\n===== seed {seed} =====")
    seed_train_val, seed_test = train_test_split(
        df_unique, test_size=0.15, random_state=seed, stratify=df_unique['label'])
    seed_train, seed_val = train_test_split(
        seed_train_val, test_size=0.176, random_state=seed, stratify=seed_train_val['label'])

    seed_train_ds = SpamDataset(tokenize(seed_train['text']), seed_train['label'])
    seed_test_ds = SpamDataset(tokenize(seed_test['text']), seed_test['label'])

    # Fresh model for each seed
    seed_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    seed_args = TrainingArguments(output_dir=f"./out_{seed}", num_train_epochs=1,
                                  per_device_train_batch_size=16, report_to="none",
                                  logging_steps=500)
    seed_trainer = Trainer(model=seed_model, args=seed_args, train_dataset=seed_train_ds)
    seed_trainer.train()

    pred = np.argmax(seed_trainer.predict(seed_test_ds).predictions, axis=-1)
    bert_scores.append((
        accuracy_score(seed_test['label'], pred),
        precision_score(seed_test['label'], pred),
        recall_score(seed_test['label'], pred),
        f1_score(seed_test['label'], pred)
    ))
    print(f"seed {seed}: "
          f"Acc {bert_scores[-1][0]:.4f} | P {bert_scores[-1][1]:.4f} | "
          f"R {bert_scores[-1][2]:.4f} | F1 {bert_scores[-1][3]:.4f}")

bert_scores = np.array(bert_scores)
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1']
print("\nDistilBERT 3-seed mean ± std:")
for i, metric in enumerate(metric_names):
    print(f"  {metric:9} {bert_scores[:, i].mean():.4f} ± {bert_scores[:, i].std():.4f}")


===== seed 0 =====


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
500,0.094815
1000,0.050089


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Step,Training Loss
500,0.101096
1000,0.035272


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

seed 2: Acc 0.9926 | P 0.9909 | R 0.9936 | F1 0.9922

DistilBERT 3-seed mean ± std:
  Accuracy  0.9928 ± 0.0006
  Precision 0.9916 ± 0.0006
  Recall    0.9933 ± 0.0015
  F1        0.9925 ± 0.0007


## Takeaways

* DistilBERT achieves approximately 0.99 across all evaluation metrics and consistently outperforms the Logistic Regression baselines across multiple seeds, particularly on precision (0.992 vs. 0.981), the metric most relevant for spam filtering.
* Although the performance gain is modest in absolute terms, its consistency across seeds suggests that the improvement reflects a genuine modeling advantage rather than random variation.
* The initially high scores were verified, not taken at face value: removing 9.6% duplicate emails and re-evaluating left performance essentially unchanged, confirming the result reflects the dataset's separability rather than data leakage.
* The trade-off is computational cost: inference takes roughly 8 ms per email compared to about 1 µs for Logistic Regression, making DistilBERT approximately 8,000× slower.
* **Verdict:** DistilBERT delivers the strongest predictive performance, but the dataset contains such strong word-level signals that a simple Logistic Regression captures most of the available information at a fraction of the computational cost. The preferred model therefore depends on whether accuracy or efficiency is the primary deployment constraint.
